<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/10_score_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, time, pandas as pd, numpy as np
BASE = '/content/drive/MyDrive/longevity_project'
hallmarks = json.load(open(f'{BASE}/data/processed/hallmarks.json'))
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))
vcap = pd.read_csv(f'{BASE}/data/processed/vcap_sig_info.csv')
genes = pd.read_csv(f'{BASE}/data/raw/GSE92742_Broad_LINCS_gene_info.txt.gz', sep='\t')

bing = genes[genes['pr_is_bing'] == 1]
id_to_symbol = dict(zip(bing['pr_gene_id'].astype(str), bing['pr_gene_symbol']))

Mounted at /content/drive


In [2]:
vcap['name'] = vcap['pert_iname'].str.lower()
vcap['is_24h'] = vcap['pert_time'] == 24
pick = vcap.sort_values(['is_24h', 'pert_dose'], ascending=False).groupby('name').head(1).copy()
pick['is_named'] = ~pick['pert_iname'].str.startswith('BRD-')
print(len(pick), 'compounds |', pick['is_named'].sum(), 'named')

15542 compounds | 2966 named


In [3]:
sigs = pd.read_parquet(f'{BASE}/data/processed/vcap_signatures.parquet', columns=list(id_to_symbol))
sigs = sigs.loc[pick['sig_id']]
sigs.columns = [id_to_symbol[c] for c in sigs.columns]
sigs = sigs.loc[:, ~sigs.columns.duplicated()]
sigs.index = pick['name'].values
print(sigs.shape)

(15542, 10174)


In [4]:
M = pd.DataFrame(0.0, index=sigs.columns, columns=list(hallmarks))
for hm, h in hallmarks.items():
    gs = [g for g in h['genes_final'] if g in M.index]
    M.loc[gs, hm] = 1 / len(gs)

t = time.time()
fast = pd.DataFrame(sigs.values @ M.values, index=sigs.index, columns=M.columns)
for hm, h in hallmarks.items():
    if h['good'] == 'down':
        fast[hm] = -fast[hm]
print(f'Scored {len(fast)} compounds in {time.time() - t:.1f} seconds')

Scored 15542 compounds in 4.0 seconds


In [5]:
from scipy.stats import spearmanr
val = pd.read_csv(f'{BASE}/results/validation_nes.csv', index_col='name')
common = val.index.intersection(fast.index)

for hm, h in hallmarks.items():
    col = f"{hm}|{h['validation']['version']}"
    gsea = -val.loc[common, col] if h['good'] == 'down' else val.loc[common, col]
    r = spearmanr(gsea, fast.loc[common, hm]).correlation
    print(f'{hm:17} agreement with GSEA: {r:.2f}')

senescence        agreement with GSEA: 0.85
nutrient_sensing  agreement with GSEA: 0.92
mitochondria      agreement with GSEA: 0.95
autophagy         agreement with GSEA: 0.73
inflammation      agreement with GSEA: 0.94
proteostasis      agreement with GSEA: 0.91


In [6]:
pct = (fast.rank(pct=True) * 100).round(1)
z = ((fast - fast.mean()) / fast.std()).round(2)

matrix = pd.concat({'score': fast.round(4), 'z': z, 'pct': pct}, axis=1)
matrix.columns = [f'{hm}_{kind}' for kind, hm in matrix.columns]
matrix['top5_count'] = (pct >= 95).sum(axis=1)
matrix['is_named'] = pick.set_index('name').loc[matrix.index, 'is_named'].values

matrix.to_parquet(f'{BASE}/results/hallmark_matrix_baseline.parquet')
matrix.to_csv(f'{BASE}/results/hallmark_matrix_baseline.csv')
print(matrix.shape)

(15542, 20)


In [7]:
for hm, k in kd['by_hallmark'].items():
    helpful = [d for d in k['helpful'] if d in pct.index]
    harmful = [d for d in k['harmful'] if d in pct.index]
    h = pct.loc[helpful, hm].mean()
    b = pct.loc[harmful, hm].mean() if harmful else np.nan
    gap = h - (b if harmful else 50)
    print(f"{hm:17} helpful avg {h:5.1f} | harmful avg {b:5.1f} | gap {gap:5.1f} "
          f"(Session 9: {hallmarks[hm]['validation']['gap']:.1f})")

senescence        helpful avg  43.0 | harmful avg  18.2 | gap  24.8 (Session 9: 11.2)
nutrient_sensing  helpful avg  69.6 | harmful avg   nan | gap  19.6 (Session 9: 19.5)
mitochondria      helpful avg  77.3 | harmful avg  45.5 | gap  31.7 (Session 9: 29.4)
autophagy         helpful avg  58.1 | harmful avg   nan | gap   8.1 (Session 9: 2.8)
inflammation      helpful avg  48.9 | harmful avg  13.3 | gap  35.6 (Session 9: 25.8)
proteostasis      helpful avg  65.6 | harmful avg  24.4 | gap  41.2 (Session 9: 32.5)


In [8]:
named = matrix[matrix['is_named']]
for hm in hallmarks:
    print(f"{hm:17}", ', '.join(named[f'{hm}_pct'].nlargest(8).index))

named = named.assign(avg_pct=named[[f'{hm}_pct' for hm in hallmarks]].mean(axis=1))
print('\nTop multi-hallmark compounds:')
print(named.sort_values(['top5_count', 'avg_pct'], ascending=False)
      [['top5_count', 'avg_pct']].head(15))

senescence        sj-172550, as-605240, lansoprazole, cay-10415, tubacin, verteporfin, u-54494a, ns-3694
nutrient_sensing  withaferin-a, aminopurvalanol-a, cct-018159, nch-51, penfluridol, midostaurin, scriptaid, sb-366791
mitochondria      nimesulide, androstenedione, erlotinib, sj-172550, indinavir, nitrazepam, xmd-1499, kenpaullone
autophagy         vorinostat, cucurbitacin-i, puromycin, pci-24781, nsc-632839, mg-132, tyrphostin-ag-1478, curcumin
inflammation      sj-172550, verteporfin, ns-3694, carbetocin, tubacin, amcinonide, as-605240, lansoprazole
proteostasis      narciclasine, sn-38, withaferin-a, irinotecan, gamma-linolenic-acid, sb-366791, cct-018159, mycophenolate-mofetil

Top multi-hallmark compounds:
              top5_count    avg_pct
oxiconazole            4  67.583333
pd-160170              3  90.583333
ucl-2077               3  88.600000
amonafide              3  77.450000
pipamperone            3  75.383333
sirolimus              3  73.533333
guanabenz              

In [9]:
from google.colab import drive
drive.mount('/content/drive')
import json, pandas as pd, numpy as np
from scipy.stats import spearmanr
BASE = '/content/drive/MyDrive/longevity_project'

hallmarks = json.load(open(f'{BASE}/data/processed/hallmarks.json'))
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))
matrix = pd.read_parquet(f'{BASE}/results/hallmark_matrix_baseline.parquet')
val = pd.read_csv(f'{BASE}/results/validation_nes.csv', index_col='name')
common = val.index.intersection(matrix.index)

agree, new_gaps, old_gaps = [], [], []
for hm, h in hallmarks.items():
    col = f"{hm}|{h['validation']['version']}"
    gsea = -val.loc[common, col] if h['good'] == 'down' else val.loc[common, col]
    agree.append(spearmanr(gsea, matrix.loc[common, f'{hm}_score']).correlation)

    k = kd['by_hallmark'][hm]
    helpful = [d for d in k['helpful'] if d in matrix.index]
    harmful = [d for d in k['harmful'] if d in matrix.index]
    hv = matrix.loc[helpful, f'{hm}_pct'].mean()
    bv = matrix.loc[harmful, f'{hm}_pct'].mean() if harmful else 50
    new_gaps.append(hv - bv)
    old_gaps.append(h['validation']['gap'])

diff = np.mean(new_gaps) - np.mean(old_gaps)
compare = 'similar to' if abs(diff) <= 5 else ('higher than' if diff > 0 else 'lower than')
K = int(((matrix['top5_count'] >= 3) & matrix['is_named']).sum())

print(f"Scored all {len(matrix):,} VCAP compounds on six hallmarks in seconds with a fast averaging method "
      f"(agreement with GSEA: {min(agree):.2f}–{max(agree):.2f} across hallmarks); known-drug gaps at full scale "
      f"{compare} Session 9; {K} named compounds rank in the top 5% on three or more hallmarks.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Scored all 15,542 VCAP compounds on six hallmarks in seconds with a fast averaging method (agreement with GSEA: 0.73–0.95 across hallmarks); known-drug gaps at full scale higher than Session 9; 128 named compounds rank in the top 5% on three or more hallmarks.
